# 01 — Ingestão da camada Bronze

## Objetivo
Ingerir o arquivo público `INTERRUPCAO_CARGA.csv` do ONS, preservando os 11 campos de negócio como texto e acrescentando metadados de rastreabilidade. A camada Bronze é a cópia estruturada da fonte para as transformações posteriores; nela não se removem duplicatas nem se corrigem valores de negócio.

**Pré-requisito:** executar `00_configuracao` e disponibilizar o CSV no Volume `ons_energia.bronze.arquivos_ons`. O resultado esperado para a versão utilizada neste MVP é de 8.326 registros e 14 colunas.

## 1. Definição da origem e do destino

São definidos o caminho do CSV no Volume e o nome completo da tabela Delta de destino. Os nomes são explícitos para que o notebook também possa ser executado de maneira independente, desde que a infraestrutura já esteja preparada.

In [0]:

from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType

CATALOG = "ons_energia"
SCHEMA = "bronze"

VOLUME_PATH = f"/Volumes/{CATALOG}/{SCHEMA}/arquivos_ons"

CSV_PATH = f"{VOLUME_PATH}/INTERRUPCAO_CARGA.csv"

TABELA_BRONZE = f"{CATALOG}.{SCHEMA}.bronze_interrupcao_carga"

print("Origem:", CSV_PATH)
print("Destino:", TABELA_BRONZE)


In [0]:

schema_bronze = StructType([
    StructField("cod_perturbacao", StringType(), True),
    StructField("din_interrupcaocarga", StringType(), True),
    StructField("id_subsistema", StringType(), True),
    StructField("nom_subsistema", StringType(), True),
    StructField("id_estado", StringType(), True),
    StructField("nom_agente", StringType(), True),
    StructField("val_cargainterrompida_mw", StringType(), True),
    StructField("val_tempomedio_minutos", StringType(), True),
    StructField("val_energianaosuprida_mwh", StringType(), True),
    StructField("flg_envolveuredebasica", StringType(), True),
    StructField("flg_envolveuredeoperacao", StringType(), True)
])


## 2. Definição do esquema bruto

Os 11 atributos originais são declarados como `StringType`. Essa decisão evita conversões automáticas de datas e decimais na ingestão e permite que a camada Silver aplique regras de transformação auditáveis.

In [0]:
schema_bronze = StructType([
    StructField("cod_perturbacao", StringType(), True),
    StructField("din_interrupcaocarga", StringType(), True),
    StructField("id_subsistema", StringType(), True),
    StructField("nom_subsistema", StringType(), True),
    StructField("id_estado", StringType(), True),
    StructField("nom_agente", StringType(), True),
    StructField("val_cargainterrompida_mw", StringType(), True),
    StructField("val_tempomedio_minutos", StringType(), True),
    StructField("val_energianaosuprida_mwh", StringType(), True),
    StructField("flg_envolveuredebasica", StringType(), True),
    StructField("flg_envolveuredeoperacao", StringType(), True)
])


## 3. Leitura do CSV

A leitura utiliza cabeçalho, separador `;`, codificação UTF-8 e o esquema explícito. O modo `FAILFAST` interrompe o processamento caso o Spark encontre registros incompatíveis com a estrutura esperada.

In [0]:

df_bronze_raw = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("sep", ";")
    .option("encoding", "UTF-8")
    .option("mode", "FAILFAST")
    .schema(schema_bronze)
    .load(CSV_PATH)
)

display(df_bronze_raw.limit(10))


## 4. Inclusão dos metadados de rastreabilidade

São adicionados `_data_ingestao` (momento da execução), `_arquivo_origem` (caminho obtido dos metadados do arquivo) e `_fonte_dados` (`ONS`). Esses campos permitem identificar a procedência dos dados sem alterar os atributos originais.

In [0]:

df_bronze = (
    df_bronze_raw
    .withColumn(
        "_data_ingestao",
        F.current_timestamp()
    )
    .withColumn(
        "_arquivo_origem",
        F.col("_metadata.file_path")
    )
    .withColumn(
        "_fonte_dados",
        F.lit("ONS")
    )
)

display(df_bronze.limit(10))


## 5. Conferência inicial

A contagem de 8.326 registros e 14 colunas é uma validação da versão específica da fonte usada neste MVP, não uma regra universal para futuras atualizações do ONS. Uma nova versão do CSV pode exigir revisão das expectativas e das análises.

In [0]:

qtd_registros = df_bronze.count()
qtd_colunas = len(df_bronze.columns)

print(f"Registros: {qtd_registros}")
print(f"Colunas: {qtd_colunas}")

assert qtd_registros == 8326, (
    f"Quantidade inesperada: {qtd_registros}"
)

assert qtd_colunas == 14, (
    f"Quantidade inesperada de colunas: {qtd_colunas}"
)

print("Validação inicial concluída!")


## 6. Persistência em Delta Lake

A tabela `ons_energia.bronze.bronze_interrupcao_carga` é gravada em formato Delta. O modo `overwrite` substitui o conteúdo anterior: esta implementação é uma carga integral de referência, não um processo incremental. A descrição da tabela é registrada no catálogo.

In [0]:

(
    df_bronze.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(TABELA_BRONZE)
)

print(f"Tabela criada: {TABELA_BRONZE}")


In [0]:

spark.sql(f"""
COMMENT ON TABLE {TABELA_BRONZE}
IS 'Dados brutos de interrupcao de carga do ONS, preservados para rastreabilidade e processamento na arquitetura Medalhao.'
""")


## 7. Validação após a gravação

A leitura da tabela persistida confirma que o volume de registros foi mantido. A consulta SQL apresenta contagem, códigos distintos e limites temporais brutos; `din_interrupcaocarga` permanece texto nesta camada. `DESCRIBE DETAIL` permite inspecionar o formato e os metadados da tabela Delta.

In [0]:

df_validacao = spark.table(TABELA_BRONZE)

print("Registros persistidos:", df_validacao.count())

assert df_validacao.count() == 8326

display(df_validacao.limit(10))


In [0]:
%sql

SELECT
    COUNT(*) AS total_registros,
    COUNT(DISTINCT cod_perturbacao) AS perturbacoes_distintas,
    MIN(din_interrupcaocarga) AS primeira_interrupcao,
    MAX(din_interrupcaocarga) AS ultima_interrupcao
FROM ons_energia.bronze.bronze_interrupcao_carga;


In [0]:
%sql

DESCRIBE DETAIL
ons_energia.bronze.bronze_interrupcao_carga;


### Resultado e encaminhamento

A Bronze contém os 8.326 registros originais e os três campos adicionais de rastreabilidade. Não há deduplicação ou descarte nesta etapa. No notebook `02_refino_silver`, os tipos serão convertidos, duplicatas integrais serão removidas e registros inválidos serão segregados em quarentena.